In [3]:
"""
12 - Quantum Random Forest
Hybrid/quantum-inspired Random Forest: uses a simple feature embedding
and trains a RandomForestClassifier. Saves metrics for comparison with
`03 - Read Dataset.ipynb`.
"""

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def compute_specificity(y_true, y_pred, average="weighted"):
    # multi-class specificity: compute per-class TN/(TN+FP) then average
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]
    supports = cm.sum(axis=1)
    specificities = []
    for i in range(n_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        spec = TN / (TN + FP) if (TN + FP) != 0 else 0
        specificities.append(spec)
    if average == "weighted":
        weights = supports / supports.sum()
        return float(np.dot(specificities, weights))
    elif average == "macro":
        return float(np.mean(specificities))
    else:
        return float(specificities)


# locate dataset in workspace
ROOT = Path.cwd()
xls_path = ROOT / "Global_Power_Emissions_Database_v1.0.xlsx"
if not xls_path.exists():
    print("Dataset not found at:", xls_path)
    print("Please place `Global_Power_Emissions_Database_v1.0.xlsx` in the same folder.")
    raise SystemExit(1)

# read and mirror preprocessing from 03 - Read Dataset.ipynb
df = pd.read_excel(xls_path, sheet_name="GPED_v1.0_Plant Level")
# some notebooks use first row as header; handle both cases
if df.iloc[0].isnull().sum() < len(df.columns):
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)

# drop Plant Name as in notebook
if "Plant Name" in df.columns:
    df = df.drop(columns=["Plant Name"])

    # Prepare features (use emissions + capacity) to avoid label leakage
    X = df[["CO2 Emissions (Mg)", "SO2 Emissions (Mg)", "PM2.5 Emissions (Mg)", "NOx Emissions (Mg)", "Total Plant Installed Capacity (MW)"]].copy()
    # convert numeric columns and drop rows with missing values in features
    X = X.apply(pd.to_numeric, errors='coerce')
    X = X.dropna()
    # target: use Fuel Types labels aligned with X's index
    if 'Fuel Types' in df.columns:
        y = df.loc[X.index, 'Fuel Types']
    else:
        y = pd.to_numeric(df.loc[X.index, 'Fuel Types Index'], errors='coerce')
    # final alignment: drop any remaining NaNs
    valid_idx = X.dropna().index.intersection(y.dropna().index)
    X = X.loc[valid_idx]
    y = y.loc[valid_idx]

# Creates new combinations of existing data features, helping the model detect complex relationships.  
# It resembles one aspect of quantum entanglement but uses entirely classical computing. 
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest (classical fallback for quantum RF)
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

metrics = {
    "model": "Quantum Random Forest (poly embed + RF)",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
    "specificity": compute_specificity(y_test, y_pred, average='weighted'),
    "recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
    "f1_score": f1_score(y_test, y_pred, average='weighted', zero_division=0),
}

results_df = pd.DataFrame([metrics])
print('\nModel results:')
print(results_df.T)

# save for comparison
results_df.to_csv(ROOT / 'quantum_rf_results.csv', index=False)
print('\nSaved results to quantum_rf_results.csv')

# Also save the confusion matrix and classification report
from sklearn.metrics import classification_report
print('\nClassification report:\n')
print(classification_report(y_test, y_pred, zero_division=0))


Model results:
                                                   0
model        Quantum Random Forest (poly embed + RF)
accuracy                                    0.934105
precision                                   0.933461
specificity                                 0.976217
recall                                      0.934105
f1_score                                    0.933175

Saved results to quantum_rf_results.csv

Classification report:

              precision    recall  f1-score   support

     BIOMASS       0.92      0.82      0.87       446
        COAL       0.92      0.92      0.92       933
          NG       0.95      0.98      0.96      1953
         OIL       0.94      0.96      0.95      2142
       OTHER       0.90      0.81      0.85       657

    accuracy                           0.93      6131
   macro avg       0.93      0.90      0.91      6131
weighted avg       0.93      0.93      0.93      6131



## Combined Model Metrics
The cell below loads the saved model comparison CSVs, builds an all-inclusive metrics table, and saves it as `all_models_metrics_table.csv`. It also inserts a small helper cell into the `03 - Read Dataset.ipynb` notebook so the metrics appear there as well. Run this cell to perform those actions.

In [5]:
import pandas as pd
from pathlib import Path
import json
ROOT = Path.cwd()
cmp_path = ROOT / 'model_comparison_with_quantum_rf.csv'
qr_path = ROOT / 'quantum_rf_results.csv'
# Load comparison data if available
df_cmp = pd.read_csv(cmp_path) if cmp_path.exists() else pd.DataFrame()
df_qr = pd.read_csv(qr_path) if qr_path.exists() else pd.DataFrame()
# Normalize QR columns to match comparison table
if not df_qr.empty:
    if 'model' in df_qr.columns:
        df_qr = df_qr.rename(columns={ 'model': 'Model', 'accuracy': 'Accuracy', 'precision': 'Precision', 'specificity': 'Specificity', 'recall': 'Sensitivity', 'f1_score': 'F1 Score'})
    if 'Time (sec)' not in df_qr.columns:
        df_qr['Time (sec)'] = pd.NA
# Ensure column set and concat
desired_cols = ['Model','Accuracy','Specificity','Precision','Sensitivity','F1 Score','Time (sec)']
def ensure_cols(df):
    for c in desired_cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df[desired_cols]
if not df_cmp.empty:
    df_cmp = ensure_cols(df_cmp)
if not df_qr.empty:
    df_qr = ensure_cols(df_qr)
all_df = pd.concat([df_cmp, df_qr], ignore_index=True, sort=False) if (not df_cmp.empty or not df_qr.empty) else pd.DataFrame(columns=desired_cols)
# Coerce numeric columns and sort
for col in ['Accuracy','Specificity','Precision','Sensitivity','F1 Score']:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce')
all_df = all_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
display(all_df)
out_path = ROOT / 'all_models_metrics_table.csv'
all_df.to_csv(out_path, index=False)
print('Saved', out_path)

# Insert helper cell into notebook 03 to load this CSV (id-less cell appended)
nb03_path = ROOT / '03 - Read Dataset.ipynb'
if nb03_path.exists():
    try:
        nb = json.loads(nb03_path.read_text(encoding='utf-8'))
    except Exception:
        # fallback: try to repair common single-object layout
        import nbformat
        nb = nbformat.read(str(nb03_path), as_version=4)
    helper_src = [
        "# Helper cell added by 12 - Quantum Random Forest: load consolidated metrics",
        "import pandas as pd",
        "from pathlib import Path",
        "ROOT = Path.cwd()",
        "metrics_path = ROOT / 'all_models_metrics_table.csv'",
        "if metrics_path.exists():",
        "    df_metrics_all = pd.read_csv(metrics_path)",
        "    print('Loaded consolidated metrics into `df_metrics_all` (in notebook 03)')",
        "    display(df_metrics_all)",
    ]
    new_cell = { 'cell_type': 'code', 'metadata': { 'language': 'python' }, 'source': helper_src }
    # Append new cell to notebook structure (handle both dict-with-cells and raw cells array)
    if isinstance(nb, dict) and 'cells' in nb:
        nb['cells'].append(new_cell)
        nb03_path.write_text(json.dumps(nb, indent=2), encoding='utf-8')
        print('Appended helper cell to 03 - Read Dataset.ipynb')
    else:
        print('Could not append helper cell: unexpected notebook format')
else:
    print('03 - Read Dataset.ipynb not found in workspace; helper not inserted')

,Model,Accuracy,Specificity,Precision,Sensitivity,F1 Score,Time (sec)
0,Random Forest,0.972156,0.983583,0.972169,0.972156,0.972093,2.947489
1,Quantum Random Forest (poly+RF),0.969173,0.980798,0.969342,0.969173,0.969050,2.993293
2,Gradient Boosting,0.955847,0.972549,0.956103,0.955847,0.955508,9.395305
3,Decision Tree,0.953461,0.974567,0.953299,0.953461,0.953354,0.146937
4,Quantum Random Forest (poly embed + RF),0.934105,0.976217,0.933461,0.934105,0.933175,<NA>
5,Logistic Regression,0.903142,0.950324,0.909469,0.903142,0.901832,2.591983
6,K-Nearest Neighbors,0.770286,0.869569,0.771799,0.770286,0.768489,0.062788
7,Support Vector Machine,0.517303,0.658699,0.531032,0.517303,0.450838,24.685163
8,Naive Bayes,0.487271,0.673243,0.524063,0.487271,0.400542,0.01778


Saved c:\Users\PATLO\OneDrive\Desktop\DATA SCIENCE\GITHUB\Data-Analytics-Studies\all_models_metrics_table.csv
Appended helper cell to 03 - Read Dataset.ipynb
